# NutriEpiDB: A Database For Dietary Compounds With Epigenetic Targets Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the NutriEpiDB dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Title: NutriEpiDB: A Database For Dietary Compounds With Epigenetic Targets
- Description: This dataset contains curated associations among 653 dietary compounds, 585 food sources, and 80 validated human epigenetic targets, supported by quantitative binding affinity measures. Data were extracted from 1,963 primary research articles via systematic searches and harmonized using chemical and biological ontologies, with compound identification standardized by PubChem and RDKit. The variables include compound properties, dietary sources, epigenetic targets, binding affinities, bibliographic information, and mechanistic annotations, providing structured information for research in nutrigenomics and epigenetic interactions.
- DOI: 10.71728/senscience.sx3s-9110
- Croissant schema: https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("\nDataset metadata:")
print(f"Title: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"DOI: {metadata['identifier']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The NutriEpiDB Croissant schema defines record sets and fields using `@id` attributes. See below for the available record sets, their field definitions, and example records. All references use the `@id` from the schema.


In [ ]:
# List the available record sets and their fields (all by @id)
record_sets = []
for rset in dataset.record_sets():
    print(f"RecordSet @id: {rset['@id']}")
    print(f"  Name: {rset.get('name','(no name)')}")
    print(f"  Description: {rset.get('description','(no desc)')}")
    print("  Fields:")
    for field in rset['fields']:
        print(f"    Field @id: {field['@id']}", end='')
        if 'name' in field:
            print(f" (name: {field['name']})")
        else:
            print()
    record_sets.append(rset['@id'])
    print()
# Show a few example records for each record set
for rset_id in record_sets:
    print(f"Sample records from RecordSet {rset_id}:")
    for i, rec in enumerate(dataset.records(record_set=rset_id)):
        if i < 2:  # Show two example records
            print(rec)
        else:
            break
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

In this example, we extract all available record sets using their `@id` and load their records into pandas DataFrames for further exploration. All fields/columns are referenced by their respective `@id`.

In [ ]:
dataframes = {}

# Load all available record sets into DataFrames
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet {record_set_id}: columns: {df.columns.tolist()}")
    print(f"Sample rows from {record_set_id}:")
    print(df.head(), "\n")

# For demonstration, select the first record set loaded for further EDA
selected_record_set_id = record_sets[0] if record_sets else None
df = dataframes[selected_record_set_id] if selected_record_set_id else pd.DataFrame()
if not df.empty:
    print(f"Selected DataFrame columns ({selected_record_set_id}):")
    print(df.columns.tolist())
    print(df.head())
else:
    print("No dataframes loaded. Check Croissant schema for available record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, outlier removal, and grouping. All operations should reference columns and fields using their `@id`.

In this section, we select a numeric field (e.g., binding affinity) and a grouping field (e.g., epigenetic target) using their `@id` values (as discovered in the schema overview).

In [ ]:
# Example EDA on selected record set DataFrame
if not df.empty:
    # Attempt to identify a numeric and group field by inspecting column names
    numeric_field_id = None
    group_field_id = None
    # Heuristics for demo: search for 'affinity' and 'target' in column names
    for col in df.columns:
        if 'affinity' in col.lower():
            numeric_field_id = col
        if 'target' in col.lower():
            group_field_id = col
    # If not found, pick the first float/integer column
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    # Pick a candidate group field
    if not group_field_id:
        for col in df.columns:
            if df[col].dtype == object:
                group_field_id = col
                break
    print(f"Numeric field for analysis: {numeric_field_id}")
    print(f"Group field: {group_field_id}")
    
    # Filtering: e.g., binding affinity > threshold
    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping if group_field exists
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
else:
    print("EDA: No DataFrame available for selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields (by their @id) in the dataset.

Below, we plot the distribution of the selected numeric field, grouped by the grouping field. All axes labels use the `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize if numeric/group fields available
if not df.empty and numeric_field_id and group_field_id:
    plt.figure(figsize=(10,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Grouped boxplot
    if group_field_id in df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=90)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization: Data not available or field selection failed.")

## 6. Conclusion
This notebook demonstrated loading the NutriEpiDB dataset using the `mlcroissant` library, viewing its record sets and fields (all by `@id`), and performing initial EDA and visualizations.

Key observations:
- Data exploration is efficient using Croissant schemas with `mlcroissant`.
- All entities and fields are referenced by their `@id`, ensuring reproducibility and transparency.
- The record sets contain rich information linking dietary compounds, food sources, epigenetic targets, and quantitative measures (such as binding affinity).
- Filtering, normalization, and grouping operations are easily performed on extracted DataFrames.
- Visualizations reveal data distributions and help identify patterns across targets and compounds.

For further work:
- Deeper analysis of compound–target relationships.
- Model building for predictive tasks.
- Integration with ontologies for semantic enrichment.

---
All exploration steps above reference entities with their `@id` as required by the Croissant schema and dataset best practices.